# CAMELS: aligned ensemble on the top 13 parameters

Runs the neural compression step as an **aligned ensemble**, to get an error bar on every latent coordinate ahead of symbolic regression.

- **Inputs:** the 13 parameters found by the screen (`notes/whittle_procedure.md`). The other 22 stay in the simulations at their prior and are marginalised.
- **Data vector:** the raw 344 standardised features.
- **Data network:** 344 → 512 → 512 → 256 → 128 → latents, GELU, with no linear shortcut from the data to the output (set in the Settings cell). The first run used 256 → 256 → 128 plus a linear shortcut.
- **What runs:** the one-step loss on those 13 parameters, then an ensemble of maps with different data-network architectures, bootstrapped and aligned to a common frame. No symbolic regression.
- **Two settings:** 13 latents (η and the data summary both 13-dimensional), and the informative rank chosen by the information rule.
- **Output**, per split:
  - `models.pkl`: the fitted ensemble, reloadable with `load_ensemble_models`, so the aligned coordinates can be evaluated at any θ afterwards.
  - `augmented_empirical.npz` and `augmented_uniform.npz`: 20000 fresh θ each, with the aligned coordinates of every member (`ys`), their mean `y` and spread `y_std`. These are the inputs for symbolic regression. *Empirical* follows the marginals of the training simulations; *uniform* covers the scaled box evenly.
  - `ensemble.npz` (the same on the 205 test simulations, plus each member's data summaries) and `record.json`.

Use a GPU runtime: *Runtime → Change runtime type → GPU*.

In [ ]:
import jax
print("JAX devices:", jax.devices())
!nvidia-smi -L || echo "no GPU visible: switch the runtime type to GPU"

## 1. Get the code

The branch `camels-whittle` holds the scripts. If the repository is private, add a GitHub token as a Colab secret named `GH_TOKEN` (key icon in the left sidebar) and give this notebook access to it.

In [ ]:
import os, subprocess
BRANCH = "camels-whittle"
REPO = "tlmakinen/degeneracy_distillery"
token = ""
try:
    from google.colab import userdata
    token = userdata.get("GH_TOKEN") or ""
except Exception:
    pass
url = f"https://{token + '@' if token else ''}github.com/{REPO}.git"
if not os.path.exists("/content/degeneracy_distillery"):
    out = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, "/content/degeneracy_distillery"],
                         capture_output=True, text=True)
    print("clone:", "ok" if out.returncode == 0 else out.stderr.replace(token, "***") if token else out.stderr)
else:
    # an existing checkout from an earlier run: bring it up to date with the branch
    out = subprocess.run(["git", "-C", "/content/degeneracy_distillery", "pull", "--ff-only", url, BRANCH],
                         capture_output=True, text=True)
    print("update:", "ok" if out.returncode == 0 else out.stderr.replace(token, "***") if token else out.stderr)
%cd /content/degeneracy_distillery
# --no-deps keeps Colab's own GPU build of JAX. pyoperon is needed only because the CAMELS
# loader imports a helper from the symbolic-regression module; no symbolic regression runs here.
!pip install -q -e . --no-deps
# -U: Colab ships a Flax that can be older than its JAX. The old Flax then fails with
# "jax.core.get_opaque_trace_state was removed".
!pip install -q -U flax optax
!pip install -q h5py pyoperon
!git log --oneline -1

In [ ]:
# Check that the installed JAX and Flax work together, in a fresh process as the runs below use.
import shutil, subprocess, sys
CHECK = """
import jax, flax, flax.linen as nn
class M(nn.Module):
    @nn.compact
    def __call__(self, x):
        return nn.Dense(3)(x)
p = M().init(jax.random.PRNGKey(0), jax.numpy.ones((2, 4)))
jax.jit(M().apply)(p, jax.numpy.ones((2, 4)))
print("jax", jax.__version__, "| flax", flax.__version__, "| device", jax.devices()[0].platform)
"""
def check():
    out = subprocess.run([sys.executable, "-c", CHECK], capture_output=True, text=True)
    print(out.stdout.strip() if out.returncode == 0 else out.stderr.strip().splitlines()[-1])
    return out.returncode == 0, "device gpu" in out.stdout
ok, on_gpu = check()
if ok and not on_gpu and shutil.which("nvidia-smi"):
    # The Flax upgrade pulled in a newer JAX without its CUDA plugin: install the matching one.
    !pip install -q -U "jax[cuda12]"
    ok, on_gpu = check()
assert ok, "JAX and Flax do not work together: send back the line printed above"
print("running on the GPU" if on_gpu else "WARNING: JAX sees no GPU, the runs below will be slow")

## 2. Upload the data

Choose `data_L50_TNG_v3.hdf5` (3.4 MB, from `data_scratch/` on your machine).

In [ ]:
from pathlib import Path
dst = Path("data_scratch/data_L50_TNG_v3.hdf5")
dst.parent.mkdir(exist_ok=True)
if not dst.exists():
    from google.colab import files
    up = files.upload()
    Path(next(iter(up))).rename(dst)
print(dst, dst.stat().st_size, "bytes")

## 3. Settings

In [ ]:
# The 13 parameters above threshold in all 8 screening runs, in rank order.
ONLY = ("Omega_m,IMFslope,Omega_b,H_0,sigma_8,n_s,A_SN1,A_SN2,WindDumpFactor,"
        "WindFreeTravelDensFac,SNII_MinMass,MinWindVel,WindEnergyReductionFactor")
K = len(ONLY.split(","))
N_SPLITS = 3        # each trial is a different train/test split
ENSEMBLE_K = 10     # ensemble members
EST_FEATURES = "512x512x256x128"   # hidden layers of the data network, 344 -> ... -> latents
EST_SKIP = "false"                 # "true" adds a linear shortcut from the data to the output
N_AUG = 20000                      # fresh theta saved per draw, with every member's aligned coordinates
OUT = "results/camels_top13_deep"
COMMON = (f'--problem camels --problem-arg n_pca=0 --problem-arg est_features={EST_FEATURES} '
          f'--problem-arg est_skip={EST_SKIP} --only "{ONLY}" --m-probe {K} '
          f'--ensemble-k {ENSEMBLE_K} --skip-sr --save-aug {N_AUG} --num-trials {N_SPLITS}')
SHOW = r'grep -E "^\[(ladder|ensemble|augment|calib|trial)\]|^=== |Traceback|Error"'
print(K, "parameters;", N_SPLITS, "splits;", ENSEMBLE_K, "members; data network", EST_FEATURES, "shortcut", EST_SKIP)

## 4a. Ensemble with 13 latents

η(θ) and the data summary are both 13-dimensional. The member spread does not show which latent directions the data constrain: all members start from the same η(θ) network, so unconstrained directions keep a small spread. Read the rank from the information per axis.

In [ ]:
!python scripts/iterative_whittle.py {COMMON} --refit-pad {K} --out-dir {OUT}/m{K} 2>&1 | {SHOW}

## 4b. Ensemble at the informative rank

The same, but the map is refitted with as many latents as clear 0.5 nats of held-out information (seven or eight in the local runs). This is the setting the sweep driver uses before symbolic regression.

In [ ]:
!python scripts/iterative_whittle.py {COMMON} --out-dir {OUT}/rank 2>&1 | {SHOW}

## 5. Summary

For each split: the rank found, the information per latent axis, and the ensemble spread per aligned coordinate relative to that coordinate's own range over the test simulations (small = well determined).

In [ ]:
import glob, json
import numpy as np

for setting in (f"m{K}", "rank"):
    print(f"\n=== {setting} ===")
    for d in sorted(glob.glob(f"{OUT}/{setting}/runs/*")):
        rec = json.load(open(f"{d}/record.json"))
        r = rec.get("discovery", {})
        z = np.load(f"{d}/ensemble.npz", allow_pickle=True)
        y, y_std = z["y"], z["y_std"]
        rel = np.median(y_std, axis=0) / (y.std(axis=0) + 1e-12)
        print(Path(d).name)
        print("  rank by information rule:", r.get("r_hat"), "| latents fitted:", r.get("m_fit"),
              "| members agreeing on the JtJ rank:", r.get("jtj_rank_agreement_k"), "of", z["ys"].shape[0])
        print("  information per axis (nats):", r.get("info_final_nats"))
        print("  ensemble spread / coordinate range, per coordinate:", np.round(rel, 3))
        print("  coordinates with spread below 10% of their range:", int((rel < 0.1).sum()), "of", len(rel))

## 6. Download the results

Send the zip back for the analysis. It holds the models, the augmentation sets, the aligned ensembles and the run records: about 250 MB with the default settings.

Check that every run folder lists `models.pkl` and both `augmented_*.npz` before downloading.

In [ ]:
!ls -la {OUT}/*/runs/*/ | grep -E "runs|models|augmented"
!zip -qr camels_top13_deep.zip {OUT}
from google.colab import files
files.download("camels_top13_deep.zip")

## Next

Symbolic regression on the aligned coordinates, weighted by `y_std`. That needs ESR and pyoperon (`notes/COLAB_SETUP.md`), and is run by dropping `--skip-sr` from the command above.